# 1. Build the corpus

Stage 0 and 1. CPU only, no model, no GPU.

The corpus is real M&A events from SEC EDGAR. The announcement 8-K gives exact ground
truth and a real dated moment when the information became public. Control entities are
matched to restricted ones on sector and size band, because asking whether you can tell
them apart needs a comparison group that resembles them.

Set `corpus.synthetic: true` in the config to build an offline corpus instead, which is
what the smoke profile does. Structure is identical, only the language is invented.

In [ ]:
# On Kaggle or Colab, uncomment to install
# !pip install -q -e /kaggle/working/silentwall
# !pip install -q -e .

from silentwall.config import load_config
from silentwall.pipeline import prepare_workspace, run_method, run_sweep, save_workspace
from silentwall.report.render import render_comparison, render_markdown, write_comparison

CONFIG = "../configs/smoke.yaml"
cfg = load_config(CONFIG)
print(cfg.profile, cfg.tier, "methods:", len(cfg.methods))

In [ ]:
ws = prepare_workspace(cfg)

## What came out

`pair_id` is shared by a restricted entity and the control it was matched to. Splits and
cross-validation folds group on that, never on the entity, because a matched pair sitting
on both sides of a fold boundary would let a model learn the matching rule instead of the
behavioural signal.

In [ ]:
print("restricted:", len(ws.corpus.restricted))
print("controls:  ", len(ws.corpus.controls))
print("pairs:     ", len(ws.corpus.pair_ids))
print("corpus hash:", ws.corpus.manifest.corpus_hash[:16])

d = ws.corpus.deals[0]
print()
print("sample deal:", d.acquirer_name, "acquires", d.target_name, "on", d.announcement_date)
for f in d.protected_fields:
    print(f"  {f.name}: {f.value_raw}  (normalized {f.value_normalized})")

## Private-side artifacts

The documents a deal team would hold. Composed from templates over already-public filing
fields, so no real confidential document is involved and the output is byte-reproducible.

In [ ]:
print("artifacts:", len(ws.artifacts))
print()
print(ws.artifacts[0].text)

## Splits

The dev half is where a containment method is allowed to calibrate. The eval half is
where the reported numbers come from. `DevSplit` carries no reference to eval entities,
so a method cannot reach one by accident.

In [ ]:
audit = ws.splits.audit()
for k, v in audit.items():
    print(f"{k}: {v}")

In [ ]:
save_workspace(ws, "../artifacts/corpus")